# 第3章 债券与利率风险

> **核心问题**：一张承诺未来付款的债券，今天为什么有价格？市场利率变化时，旧债券的价格为什么会反向变化？

- 金融线：票息、面值、到期、收益率、利率/信用/流动性风险。
- 数学线：贴现求和、加权平均、一阶与二阶敏感度。
- Python线：现金流数组、数值求根、函数分解、交互曲线和近似误差。

## AI学习状态

当前进度：第3章开始  
已掌握：现值、NPV和数组贴现  
仍然薄弱：待填写  
下一步：先区分票面利率、市场收益率和债券价格。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 3.1 债券是一组带条件的未来付款承诺

购买债券通常意味着把资金借给发行者。简化固定利率债券包含：

- 面值 $F$：到期偿还的本金；
- 年票面利率 $c$：决定每年票息 $C=F\times c$；
- 到期期限 $T$；
- 市场到期收益率 $y$：市场用于贴现这组现金流的利率。

按年付息时：

$$P=\sum_{t=1}^{T}\frac{C}{(1+y)^t}+\frac{F}{(1+y)^T}$$

Investor.gov将债券解释为类似借据的债务证券，并提醒利率变化会影响债券价值；中国市场的具体产品与规则应继续查阅交易所和监管机构资料。

In [ ]:
face = 1_000
coupon_rate = 0.05
maturity = 3
yield_rate = 0.04

coupon = face * coupon_rate
cash_flows = np.full(maturity, coupon, dtype=float)
cash_flows[-1] += face
times = np.arange(1, maturity + 1)
present_values = cash_flows / (1 + yield_rate) ** times

bond_table = pd.DataFrame({"年份": times, "现金流": cash_flows, "现值": present_values})
display(bond_table.style.format({"现金流": "{:,.2f}", "现值": "{:,.2f}"}))
print("债券价格：", round(present_values.sum(), 2))

### 运行前后的解释

票面利率5%而市场收益率4%时，债券价格为什么高于面值？请用“旧债券承诺的票息”和“市场要求的收益率”解释，不要只说公式结果。

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

In [ ]:
def bond_price(face, coupon_rate, maturity, yield_rate, frequency=1):
    # 固定利率债券价格；期限以年计且应与付息频率相容。
    periods = int(round(maturity * frequency))
    coupon = face * coupon_rate / frequency
    cash_flows = np.full(periods, coupon, dtype=float)
    cash_flows[-1] += face
    period_yield = yield_rate / frequency
    times = np.arange(1, periods + 1)
    return float(np.sum(cash_flows / (1 + period_yield) ** times))


for y in [0.03, 0.05, 0.07]:
    print(f"市场收益率{y:.0%} -> 价格{bond_price(1000, 0.05, 5, y):.2f}")

**三个不要混淆的量**

- 票面利率决定合同票息，发行后通常固定；
- 市场收益率随市场、期限和信用条件变化；
- 当前价格是未来现金流按市场收益率贴现的结果。

当票面利率=市场收益率时，简化债券价格等于面值；前提是付息与利率口径一致且没有额外条款。

## 3.2 价格—收益率曲线

其他条件不变，市场收益率上升，固定现金流的现值下降；因此债券价格通常下降。这是现金流贴现关系，不是“债券市场的神秘规则”。

In [ ]:
yields = np.linspace(0.001, 0.12, 200)
maturities = [1, 5, 10, 20]

fig, ax = plt.subplots()
for t in maturities:
    prices = [bond_price(1000, 0.05, t, y) for y in yields]
    ax.plot(yields, prices, label=f"{t}年期")
ax.axhline(1000, color="gray", linestyle="--")
ax.set(title="固定票息债券的价格—收益率关系", xlabel="市场收益率", ylabel="债券价格（元）")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend(); plt.show()

### 观察问题

1. 哪种期限的曲线更陡？这意味着什么？
2. 曲线是直线吗？
3. 收益率从2%升到3%和从10%升到11%，价格变化是否完全相同？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 3.3 久期：现金流时间与价格敏感度

Macaulay久期是现金流时点按其现值权重计算的平均时间：

$$D_M=\frac{\sum_t t\cdot PV(CF_t)}{P}$$

修正久期 $D_{mod}=D_M/(1+y)$ 给出小幅收益率变化下的价格近似：

$$\frac{\Delta P}{P}\approx-D_{mod}\Delta y$$

负号表达价格与收益率通常反向变化。

In [ ]:
def bond_duration(face, coupon_rate, maturity, yield_rate):
    coupon = face * coupon_rate
    cash_flows = np.full(maturity, coupon, dtype=float)
    cash_flows[-1] += face
    times = np.arange(1, maturity + 1)
    pvs = cash_flows / (1 + yield_rate) ** times
    price = pvs.sum()
    macaulay = np.sum(times * pvs) / price
    modified = macaulay / (1 + yield_rate)
    return price, macaulay, modified


price0, macaulay, modified = bond_duration(1000, 0.05, 10, 0.05)
print({"价格": round(price0, 2), "Macaulay久期": round(macaulay, 4), "修正久期": round(modified, 4)})

**数学连接：加权平均**

久期不是简单的到期年数。早期票息降低现金流的平均等待时间；零息债券只有到期一笔现金流，其Macaulay久期等于到期期限。

In [ ]:
shocks = np.array([-0.02, -0.01, -0.0025, 0.0025, 0.01, 0.02])
rows = []
for shock in shocks:
    exact_price = bond_price(1000, 0.05, 10, 0.05 + shock)
    duration_price = price0 * (1 - modified * shock)
    rows.append({"收益率变化": shock, "精确价格": exact_price,
                 "久期近似": duration_price, "近似误差": duration_price - exact_price})
duration_check = pd.DataFrame(rows)
display(duration_check.style.format({"收益率变化": "{:+.2%}", "精确价格": "{:,.2f}",
                                     "久期近似": "{:,.2f}", "近似误差": "{:+.2f}"}))

### 观察问题

久期近似在小冲击还是大冲击时更准确？误差为什么表现出弯曲关系？这将引出凸性。

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 3.4 凸性：用二阶项修正曲线弯曲

久期是一阶近似；价格—收益率曲线弯曲时，可加入凸性项：

$$\frac{\Delta P}{P}\approx-D_{mod}\Delta y+\frac{1}{2}Convexity(\Delta y)^2$$

本章用数值差分估计二阶敏感度，避免一次引入过多债券市场计数规则。

In [ ]:
def numerical_duration_convexity(price_function, y, step=1e-4):
    p0 = price_function(y)
    p_up = price_function(y + step)
    p_down = price_function(y - step)
    modified_duration = -(p_up - p_down) / (2 * step * p0)
    convexity = (p_up - 2 * p0 + p_down) / (step**2 * p0)
    return modified_duration, convexity


price_fn = lambda y: bond_price(1000, 0.05, 10, y)
d_num, convexity = numerical_duration_convexity(price_fn, 0.05)
print({"数值修正久期": round(d_num, 4), "数值凸性": round(convexity, 4)})

In [ ]:
exact = np.array([price_fn(0.05 + s) for s in shocks])
duration_only = price0 * (1 - d_num * shocks)
duration_convexity = price0 * (1 - d_num * shocks + 0.5 * convexity * shocks**2)

plt.plot(shocks, exact, "o-", label="精确重定价")
plt.plot(shocks, duration_only, "--", label="仅久期")
plt.plot(shocks, duration_convexity, ":", linewidth=3, label="久期+凸性")
plt.xlabel("收益率变化"); plt.ylabel("价格（元）"); plt.title("敏感度近似与精确重定价")
plt.legend(); plt.show()

## 3.5 到期收益率：从价格反求利率

市场给出价格时，可以数值求解使贴现现金流等于价格的收益率。它综合了当前价格和合同现金流，但依赖持有到期、再投资等解释条件。

In [ ]:
from scipy.optimize import brentq

market_price = 950
solved_yield = brentq(lambda y: bond_price(1000, 0.05, 5, y) - market_price, -0.9, 2.0)
print(f"价格{market_price}元对应的到期收益率：{solved_yield:.4%}")
print("代回价格：", round(bond_price(1000, 0.05, 5, solved_yield), 6))

## 3.6 不只有利率风险

债券还可能面临：发行人不能按约付款的信用风险、难以及时成交的流动性风险、通胀侵蚀固定现金流购买力、提前赎回等合同条款风险。

一个高票息债券不一定“更好”：更高票息可能对应更高信用风险，价格也可能已经反映风险。

In [ ]:
promised = 1_050
recovery = 400
default_probabilities = np.linspace(0, 0.30, 61)
expected_payments = (1 - default_probabilities) * promised + default_probabilities * recovery

plt.plot(default_probabilities, expected_payments)
plt.xlabel("教学假设：违约概率"); plt.ylabel("期望一年后付款（元）")
plt.title("信用风险如何改变期望现金流")
plt.show()

**量化编程警告**：上图只用两个结局，且把违约概率和回收额当作已知。现实信用建模必须处理估计误差、相关违约、迁徙和时间变化。

## 3.7 交互实验：自行设计一只简化债券

In [ ]:
def bond_lab(coupon_rate=0.05, maturity=10, market_yield=0.05):
    price = bond_price(1000, coupon_rate, maturity, market_yield)
    _, macaulay, modified = bond_duration(1000, coupon_rate, maturity, market_yield)
    print({"coupon_rate": coupon_rate, "maturity": maturity, "market_yield": market_yield,
           "price": round(price, 2), "modified_duration": round(modified, 3)})
    ys = np.linspace(max(0.001, market_yield - 0.04), market_yield + 0.04, 100)
    plt.plot(ys, [bond_price(1000, coupon_rate, maturity, y) for y in ys])
    plt.axvline(market_yield, color="red", linestyle="--")
    plt.xlabel("市场收益率"); plt.ylabel("价格"); plt.title("价格—收益率局部曲线"); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(bond_lab,
             coupon_rate=FloatSlider(value=.05, min=0, max=.12, step=.01, description="票面利率"),
             maturity=IntSlider(value=10, min=1, max=30, description="期限"),
             market_yield=FloatSlider(value=.05, min=.005, max=.15, step=.005, description="市场收益率"))
except ImportError:
    bond_lab()

## 3.8 编程练习：补全半年付息债券定价

输入均以年为单位；`frequency=2`表示半年付息。注意每期票息、每期收益率和总期数都必须转换。

In [ ]:
def student_bond_price(face, coupon_rate, maturity, yield_rate, frequency=2):
    # TODO：完成半年或其他频率付息债券的价格计算
    return None

In [ ]:
answer = student_bond_price(1000, 0.06, 2, 0.06, frequency=2)
if answer is None:
    print("练习尚未完成。")
else:
    print("平价债券测试通过：", np.isclose(answer, 1000, atol=1e-8))

### 我的解释

为什么长期债券通常比短期债券对收益率变化更敏感？票息提高又会怎样影响久期？

<!-- 在这里填写；完成前AI不要代答 -->

### AI批改区

<!-- 检查现金流时点、频率换算、价格方向和边界条件。 -->

## 本章总结与小项目

制作一个债券压力测试器：比较2年、5年和10年固定利率债券；设置收益率±50/100/200个基点冲击；同时报告精确重定价、久期近似和误差；单独列出无法由久期覆盖的信用与流动性风险。

**参考**：Investor.gov Bonds；上海证券交易所投资者教育“债券专题”。